In [ ]:
import numpy as np
import pandas as pd
from scipy.signal import welch


PSD feature extraction complete. Features saved to 'PSD_features.csv'


In [ ]:

def compute_psd(window, fs=256):
    """Compute Power Spectral Density (PSD) using Welch's method for all EEG channels."""
    psd_features = []
    for i in range(window.shape[1]):  # Iterate over EEG channels
        f, Pxx = welch(window[:, i], fs=fs, nperseg=fs, window='hann', scaling='density')
        psd_features.append(Pxx)  # Store the full PSD for each channel
    return np.array(psd_features)  # Shape: (num_channels, num_frequencies)


In [ ]:

def sliding_window_psd(eeg_data, outcomes, window_size, step_size, fs=256):
    """Extract PSD features using a sliding window approach."""
    all_psd_features = []
    targets = []
    n_samples, num_channels = eeg_data.shape
    
    for start in range(0, n_samples - window_size + 1, step_size):
        end = start + window_size
        window = eeg_data[start:end]
        outcome_window = outcomes[start:end]

        # Compute PSD for this window
        psd_values = compute_psd(window, fs)  # Shape: (num_channels, num_frequencies)

        all_psd_features.append(psd_values.flatten())  # Flatten to 1D per window
        targets.append(1 if np.any(outcome_window) else 0)  # Assign target based on Outcome
    
    return np.array(all_psd_features), np.array(targets)


In [ ]:

# Main processing
if __name__ == "__main__":
    # Load EEG data
    data = pd.read_csv('/Users/puchku-home/Study/PROJECT/EEG/EEG Assets/chbmit_preprocessed_data.csv')
    eeg_columns = [col for col in data.columns if col != 'Outcome']

    # Convert to NumPy arrays
    eeg_data = np.asarray(data[eeg_columns].values, dtype=np.float32)
    outcomes = np.asarray(data['Outcome'].values, dtype=np.float32)

    # Windowing parameters
    fs = 256  # Sampling frequency
    window_size = fs * 1  # 1-second window
    step_size = window_size // 2  # 50% overlap

    # Compute PSD features
    psd_features, targets = sliding_window_psd(eeg_data, outcomes, window_size, step_size, fs)

    # Convert to DataFrame and save
    num_frequencies = psd_features.shape[1] // len(eeg_columns)
    psd_feature_names = [f"{col}_f{i}" for col in eeg_columns for i in range(num_frequencies)]
    psd_df = pd.DataFrame(psd_features, columns=psd_feature_names)
    psd_df['target'] = targets
    psd_df.to_csv('/Users/puchku-home/Downloads/Frequency Feature  Generalised/PSD_features.csv', index=False)
    print("PSD feature extraction complete. Features saved to 'PSD_features.csv'")
